In [1]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

In [2]:
results_1l = pd.read_excel("resultados-1l-v2.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
    # [results_1l, results_2l, results_3l],
    # ignore_index=True
# )
results = results_1l


In [3]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_SuperZZ1_theta,MSE_SuperZZ1_theta,R2_SuperZZ2_theta,MSE_SuperZZ2_theta,...,R2_ZZx2_theta,MSE_ZZx2_theta,R2_ZZxReto_theta,MSE_ZZxReto_theta,R2_ZZy1_theta,MSE_ZZy1_theta,R2_ZZy2_theta,MSE_ZZy2_theta,R2_semiCirc_theta,MSE_semiCirc_theta
0,model_arch1_r0.01_Ld0.3_Lp0.7_seed1777,[1],0.3,0.7,0.01,1777,0.559971,0.362001,-0.499142,0.384362,...,-5.134324,-0.058463,-1.379522,0.199475,-22.144087,-0.087272,0.867178,0.381503,-6.448856,0.229919
1,model_arch1_r0.01_Ld0.3_Lp0.7_seed1725,[1],0.3,0.7,0.01,1725,0.740029,0.490495,-0.525042,0.479324,...,-4.381372,0.149940,-0.179039,0.395037,-18.473606,0.084252,0.864613,0.501536,-11.013806,0.196447
2,model_arch1_r0.01_Ld0.3_Lp0.7_seed9855,[1],0.3,0.7,0.01,9855,0.593020,0.381860,-0.833514,0.390051,...,-4.853548,-0.021423,-1.287428,0.228667,-22.434176,-0.065298,0.836441,0.392731,-7.282284,0.220549
3,model_arch1_r0.01_Ld0.3_Lp0.7_seed2557,[1],0.3,0.7,0.01,2557,0.719284,0.469794,-1.338685,0.460178,...,-3.595953,0.145139,-0.709242,0.372929,-18.744619,0.068684,0.674873,0.483294,-8.300288,0.244449
4,model_arch1_r0.01_Ld0.3_Lp0.7_seed9383,[1],0.3,0.7,0.01,9383,0.729280,0.480553,-1.116341,0.477071,...,-3.769685,0.147212,-0.683659,0.383282,-20.527321,0.058899,0.772739,0.497711,-9.762544,0.213811
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2003,model_arch100_r0.9_Ld0.7_Lp0.3_seed2389,[100],0.7,0.3,0.90,2389,0.812367,0.663924,0.169971,0.610067,...,-0.048562,0.490202,-0.141753,0.428544,-22.245726,0.074775,0.884078,0.646122,-6.413828,0.303443
2004,model_arch100_r0.9_Ld0.7_Lp0.3_seed1595,[100],0.7,0.3,0.90,1595,0.642800,0.524678,0.649288,0.588382,...,-1.006661,0.303819,-2.133339,0.444302,-36.476651,0.005520,0.830791,0.582870,-9.770060,0.184033
2005,model_arch100_r0.9_Ld0.7_Lp0.3_seed6927,[100],0.7,0.3,0.90,6927,0.761279,0.617803,0.409426,0.563302,...,-0.457326,0.450719,-0.567435,0.286065,-40.255696,-0.260936,0.886203,0.581441,-9.726722,0.208211
2006,model_arch100_r0.9_Ld0.7_Lp0.3_seed4518,[100],0.7,0.3,0.90,4518,0.861706,0.630032,-0.851324,0.543770,...,0.456724,0.497790,0.119430,0.469090,-36.329814,-0.087767,0.742281,0.572918,-19.752072,-0.025143


In [4]:
# 🔹 categorização dos sets (baseada nos comentários originais)

SETS_CATEGORY = {
    "SuperZZ1":  "Train",
    "SuperZZ2":  "Val",
    "ZZx1":      "Test",
    "ZZx2":     "Test",
    "ZZy1":     "Test",
    "ZZy2":     "Test",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZxReto":  "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33

for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"]
        - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
1290,model_arch65_r0.01_Ld0.7_Lp0.3_seed1777,[65],0.184053,0.441324,-2.348630,-0.863787
1278,model_arch64_r0.9_Ld0.7_Lp0.3_seed2557,[64],0.346848,0.514118,-2.827269,-1.082543
1238,model_arch62_r0.9_Ld0.7_Lp0.3_seed2557,[62],0.485376,0.403297,-2.852043,-1.085381
1374,model_arch69_r0.01_Ld0.7_Lp0.3_seed9383,[69],0.491950,0.242671,-2.631023,-1.086160
1750,model_arch88_r0.01_Ld0.7_Lp0.3_seed1777,[88],0.789278,0.366178,-3.066608,-1.141782



📊 MÉTRICAS COMPLETAS - TOP 5 (theta)


,model,Neurons,R2_SuperZZ1_theta,R2_SuperZZ2_theta,R2_ZZx1_theta,R2_ZZx2_theta,R2_ZZy1_theta,R2_ZZy2_theta,R2_LSG_1_theta,R2_LSG_2_theta,R2_ZZx1_inv_theta,R2_ZZxReto_theta,R2_semiCirc_theta,R2_train_mean,R2_val_mean,R2_test_mean,Score
1290,model_arch65_r0.01_Ld0.7_Lp0.3_seed1777,[65],0.184053,0.441324,-1.483604,-1.039464,-10.097170,-0.205248,-3.332418,-1.964447,-0.134581,-0.846249,-2.034487,0.184053,0.441324,-2.348630,-0.863787
1278,model_arch64_r0.9_Ld0.7_Lp0.3_seed2557,[64],0.346848,0.514118,-1.456940,-0.383463,-13.340220,0.445898,-2.312939,-1.142971,0.704150,-0.561131,-7.397804,0.346848,0.514118,-2.827269,-1.082543
1238,model_arch62_r0.9_Ld0.7_Lp0.3_seed2557,[62],0.485376,0.403297,-1.597950,0.523492,-10.880625,0.871106,-2.537910,-1.450038,0.858493,-0.730546,-10.724410,0.485376,0.403297,-2.852043,-1.085381
1374,model_arch69_r0.01_Ld0.7_Lp0.3_seed9383,[69],0.491950,0.242671,-0.719272,0.362165,-10.609386,0.845919,-1.580746,-0.799663,0.729107,0.004217,-11.911546,0.491950,0.242671,-2.631023,-1.086160
1750,model_arch88_r0.01_Ld0.7_Lp0.3_seed1777,[88],0.789278,0.366178,-0.143027,0.634348,-12.086429,0.912602,-1.794698,-0.362134,-1.674540,-0.026296,-13.059301,0.789278,0.366178,-3.066608,-1.141782


In [5]:
final_table.to_excel("BestModels-1l.xlsx")